# P03 – Measuring Fairness

**How this notebook works**
- Data loading, the check runner and the plotting helpers are **given**.
- Every function marked `# TODO` is yours. It raises `NotImplementedError` until you write it.
- After each function, run `run_checks()`. It tests your functions on the **lecture examples** (not on the lab data), so a green check means the logic is right, not that you've seen the answer.
- The `Your answer` cells are for the interpretation. That's what you'll present on Wednesday, so write it in your own words.

Theory: `Decision Systems & Performance Metrics` and `Fairness Metrics` in the vault.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path(".")  # folder containing the three CSV files

dec   = pd.read_csv(DATA_DIR / "decision_evaluation.csv")           # p, y
loan  = pd.read_csv(DATA_DIR / "loan_applicants_biased_scores.csv") # p, p_hat, g, Y
compas = pd.read_csv(DATA_DIR / "compas_preprocessed.csv")          # race, decile_score, high_risk, two_year_recid

for name, df in [("dec", dec), ("loan", loan), ("compas", compas)]:
    print(f"{name}: {df.shape}  columns={list(df.columns)}")

## Checks (given)
Run `run_checks()` whenever you've written a function. Unwritten functions show as `not implemented yet`.

In [ ]:
def _close(a, b, tol=1e-3):
    return abs(a - b) < tol

def run_checks():
    # Lecture example (week 3, slides 21 + 28): 20 loan applicants, threshold 0.7
    p   = [.81,.80,.79,.77,.75,.74,.74,.73,.72,.72,.68,.68,.68,.62,.60,.55,.48,.47,.40,.37]
    y   = [0,1,1,1,1,1,1,1,0,1,1,1,1,0,1,0,1,0,1,0]
    sex = list("mmmmfmmmffmmfffmffff")
    results = []

    def check(name, fn):
        try:
            fn(); results.append((name, "ok"))
        except NotImplementedError:
            results.append((name, "not implemented yet"))
        except AssertionError as e:
            results.append((name, f"FAILED: {e}"))
        except Exception as e:
            results.append((name, f"ERROR: {type(e).__name__}: {e}"))

    def t_matrix():
        m = decision_matrix(pd.Series(p), pd.Series(y), 0.7)
        assert m.shape == (2, 2), f"expected a 2x2 matrix, got shape {m.shape}"
        assert list(m.index) == [0, 1] and list(m.columns) == [0, 1], "index must be D=[0,1], columns Y=[0,1]"
        assert m.loc[0, 0] == 4 and m.loc[0, 1] == 6 and m.loc[1, 0] == 2 and m.loc[1, 1] == 8, \
            f"lecture example should give [[4,6],[2,8]], got {m.values.tolist()}"
        empty = decision_matrix(pd.Series([.1, .2]), pd.Series([0, 1]), 0.7)
        assert empty.shape == (2, 2) and empty.loc[1].sum() == 0, "matrix must stay 2x2 even if nobody gets D=1"

    def t_normalize():
        m = pd.DataFrame([[4, 6], [2, 8]], index=[0, 1], columns=[0, 1])
        n = normalize(m)
        assert _close(n.values.sum(), 1), "probabilities must sum to 1"
        assert _close(n.loc[0, 1], 0.3) and _close(n.loc[1, 0], 0.1), f"expected [[.2,.3],[.1,.4]], got {n.values.tolist()}"

    def t_metrics():
        men = pd.DataFrame([[1, 2], [1, 6]], index=[0, 1], columns=[0, 1])  # men, slide 28
        r = metrics(men)
        exp = dict(BR=.8, ACC=.7, PR=.7, TPR=.75, FNR=.25, FPR=.5, TNR=.5, PPV=6/7, FDR=1/7, NPV=1/3, FOR=2/3)
        for k, v in exp.items():
            assert k in r, f"missing key {k}"
            assert _close(r[k], v), f"{k}: expected {v:.3f}, got {r[k]:.3f}"

    def t_by_group():
        t = metrics_by_group(pd.Series(p), pd.Series(y), pd.Series(sex), 0.7)
        assert set(t.index) == {"m", "f"}, "one row per group (index = group value)"
        for g, pr, tpr, ppv in [("m", .7, .75, 6/7), ("f", .3, 1/3, 2/3)]:
            assert _close(t.loc[g, "PR"], pr) and _close(t.loc[g, "TPR"], tpr) and _close(t.loc[g, "PPV"], ppv), \
                f"group {g}: expected PR={pr:.2f}, TPR={tpr:.2f}, PPV={ppv:.2f}"
        # extreme example from the Socratic session: A all score 7 (7 of 10 re-offend), B all score 2 (2 of 10)
        s = pd.Series([7] * 10 + [2] * 10); yy = pd.Series([1] * 7 + [0] * 3 + [1] * 2 + [0] * 8)
        gg = pd.Series(["A"] * 10 + ["B"] * 10)
        t2 = metrics_by_group(s, yy, gg, 5)
        assert _close(t2.loc["A", "FPR"], 1.0) and _close(t2.loc["B", "FPR"], 0.0), "extreme example: FPR A=1, B=0"

    def t_calibration():
        score = pd.Series([.05] * 10 + [.15] * 5 + [.95] * 10)
        yy = pd.Series([1, 1] + [0] * 8 + [1] * 2 + [0] * 3 + [1] * 9 + [0])
        c = calibration_table(score, yy, n_bins=10)
        for col in ["mean_score", "mean_y", "n"]:
            assert col in c.columns, f"missing column {col}"
        assert len(c) == 3, f"expected 3 non-empty bins, got {len(c)}"
        assert c["n"].sum() == 25, "every individual must land in exactly one bin"
        c = c.sort_values("mean_score")
        assert np.allclose(c["mean_y"].values, [.2, .4, .9]), f"mean_y should be [.2,.4,.9], got {c['mean_y'].round(3).tolist()}"

    check("decision_matrix", t_matrix)
    check("normalize", t_normalize)
    check("metrics", t_metrics)
    check("metrics_by_group", t_by_group)
    check("calibration_table", t_calibration)
    w = max(len(n) for n, _ in results)
    for n, s in results:
        print(f"{n:<{w}}  {s}")

## Plotting helpers (given)

In [ ]:
def plot_calibration(tables, title, ax=None):
    """tables: dict {label: DataFrame from calibration_table}. Draws observed vs. predicted plus the diagonal."""
    ax = ax or plt.subplots(figsize=(5, 5))[1]
    ax.plot([0, 1], [0, 1], "--", color="grey", label="perfectly calibrated")
    for label, t in tables.items():
        ax.plot(t["mean_score"], t["mean_y"], "o-", label=label)
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="predicted probability (mean score in bin)",
           ylabel="observed share with Y=1", title=title)
    ax.legend()
    return ax

def plot_hist_by_group(values, groups, bins, title, xlabel, ax=None, density=False):
    """Overlaid histograms of `values`, one per value of `groups`."""
    ax = ax or plt.subplots(figsize=(6, 4))[1]
    for g in sorted(pd.unique(groups)):
        ax.hist(values[groups == g], bins=bins, alpha=0.5, label=str(g), density=density)
    ax.set(title=title, xlabel=xlabel, ylabel="density" if density else "count")
    ax.legend()
    return ax

---
## Task 1 – Decision matrix (0.5 pt)
10,000 individuals with true probability `p` and outcome `y`. Rule: **D = 1 if p > 0.7**.
Compute the decision matrix in absolute numbers and as probabilities.

In [ ]:
def decision_matrix(score, y, threshold):
    """Decision matrix for the rule D = 1 if score > threshold, else D = 0.

    Returns a 2x2 DataFrame of counts N_ij with
        index   = D in [0, 1]
        columns = Y in [0, 1]
    so m.loc[d, y_val] is the number of individuals with D=d and Y=y_val.
    Must stay 2x2 even if a cell is empty (fill with 0).
    """
    # TODO
    raise NotImplementedError


def normalize(matrix):
    """Counts N_ij -> probabilities p_ij = N_ij / N (same shape and labels)."""
    # TODO
    raise NotImplementedError

In [ ]:
# run_checks()
# m1 = decision_matrix(dec["p"], dec["y"], 0.7); display(m1); display(normalize(m1))

**Your answer (Task 1):** short description of the method + both matrices.

---
## Task 2 – Conditional probabilities (0.5 pt)
Matrix: $p_{00}=0.2,\ p_{01}=0.3,\ p_{10}=0.1,\ p_{11}=0.4$ (rows D, columns Y).

**Do this by hand first** with $P(A \mid B) = \frac{P(A \cap B)}{P(B)}$ and write the calculation for each metric, because the report needs the method. Then write `metrics()` below and use it to check your hand calculation. You'll reuse `metrics()` in Tasks 3 and 4.

| Metric | Formula | Calculation | Result |
|---|---|---|---|
| (a) TPR $P(D=1 \mid Y=1)$ | | | |
| (b) FPR $P(D=1 \mid Y=0)$ | | | |
| (c) TNR $P(D=0 \mid Y=0)$ | | | |
| (d) FNR $P(D=0 \mid Y=1)$ | | | |
| (e) PPV $P(Y=1 \mid D=1)$ | | | |
| (f) FDR $P(Y=0 \mid D=1)$ | | | |
| (g) NPV $P(Y=0 \mid D=0)$ | | | |
| (h) FOR $P(Y=1 \mid D=0)$ | | | |

In [ ]:
def metrics(m):
    """All metrics of a 2x2 decision matrix m (counts or probabilities; index D, columns Y).

    Returns a dict with keys:
        BR, ACC, PR, TPR, FNR, FPR, TNR, PPV, FDR, NPV, FOR
    Use float('nan') where a denominator is 0.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# run_checks()
# m2 = pd.DataFrame([[0.2, 0.3], [0.1, 0.4]], index=[0, 1], columns=[0, 1])
# metrics(m2)

---
## Task 3 – Group bias in predictions (1.5 pt)
20,000 loan applicants: true probability `p`, predicted `p_hat`, gender `g`, outcome `Y`.

In [ ]:
def calibration_table(score, y, n_bins=10):
    """Calibration table with n_bins equally spaced bins on [0, 1].

    Returns a DataFrame with one row per NON-EMPTY bin and the columns
        mean_score : average score in the bin            (x-axis)
        mean_y     : share with Y=1 in the bin, N1/N     (y-axis)
        n          : number of individuals in the bin
    """
    # TODO
    raise NotImplementedError

In [ ]:
# run_checks()
# (a) true p, all applicants
# plot_calibration({"all": calibration_table(loan["p"], loan["Y"])}, "(a) true p – all applicants");

In [ ]:
# (b) true p, separately per gender

**Your answer (3a, 3b):** what you see, and why calibration *must* hold here.

In [ ]:
# (c) p_hat, ignoring gender + the three base rates:
#     from true p, empirical (mean Y), predicted (mean p_hat)

**Your answer (3c):** does the plot look as good as with the true p? Do the base rates match? In your own words: what does a value of `p_hat` say about the people who get it?

In [ ]:
# (d) p_hat, separately per gender + group-wise base rates

**Your answer (3d):** findings, and what thresholding on `p_hat` would mean for fairness.

In [ ]:
# (e) histograms of p per group, then of p_hat per group (plot_hist_by_group)

**Your answer (3e):** do the groups differ in their true composition? What does the prediction do to it?

In [ ]:
def metrics_by_group(score, y, group, threshold):
    """Apply D = 1 if score > threshold and compute metrics() separately per group.

    Returns a DataFrame: one row per group value (index = group value),
    one column per metric key from metrics().
    """
    # TODO (hint: reuse decision_matrix and metrics)
    raise NotImplementedError

In [ ]:
# run_checks()
# (f) D = 1 if p_hat > 0.7 -> compare PR, TPR, PPV per gender

**Your answer (3f):** what do PR, TPR and PPV tell you about the fairness of this decision system? Which fairness criteria are violated?

---
## Task 4 – The COMPAS case (1.5 pt)
Use only `race == "African-American"` and `race == "Caucasian"`.
- `decile_score`: 1–10 (10 = highest predicted risk of re-arrest)
- `high_risk`: 1 for scores 5–10, 0 for 1–4
- `two_year_recid`: Y (re-arrested within 2 years)

In [ ]:
cmp = compas[compas["race"].isin(["African-American", "Caucasian"])].copy()
cmp["race"].value_counts()

In [ ]:
# (a) histogram of decile_score per race (hint: bins=np.arange(0.5, 11.5, 1) puts one bar on each score)

**Your answer (4a):**

In [ ]:
# (b) base rate of two_year_recid per race

**Your answer (4b):** if the score reflected the true probability, what would this say about the two groups?

In [ ]:
# (c) calibration plot for all defendants, with p = decile_score / 10
# Think first: the score only takes 10 values. Do you need bins at all, and would
# calibration_table's bin edges (0.1, 0.2, …) cause trouble with values like 0.1 and 1.0?

**Your answer (4c):** what can we say about the COMPAS score?

In [ ]:
# (d) calibration per race

**Your answer (4d):** calibration *between groups*? Keep the finite sample size in mind. Did the developers do a good job?

In [ ]:
# (e) + (f) D = 1 if high risk (decile_score >= 5). FPR and FNR per race (metrics_by_group)

**Your answer (4e):** explain the FPR difference so that a judge without a technical background understands it.

**Your answer (4f):** FNR difference, interpreted the same way.

**Optional:** how do (d) and (e) fit together? (→ incompatibility, base rates)